In [1]:
from pathlib import Path
import json
from pathlib import Path

from utils.comparer.evaluator import *


In [2]:
def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def flatten_label_json(label_json: dict) -> list[dict]:
    tests = []
    for section in label_json["tests"]:
        for test in section["tests"]:
            test["section"] = section["section"]
            tests.append(test)
    return tests


In [3]:
label_path = Path(f"data/labels/lablabel4.json")

label_json = load_json(label_path)
label_tests = flatten_label_json(label_json)
# label_tests 

In [5]:
evaluator = LabReportEvaluator(
    paths=["test_name", "result", "unit", "reference_value", "section"]
)


result = evaluator.evaluate([{"test_name": "wbc","result": 12}], [{"test_name": "wbc","result": "12.0"}])
result

{'avg_score': 100.0,
 'items': [{'test_name': 'wbc', 'result': '12.0', 'score': 100.0}]}

In [6]:
def calc_score(item_no, base_path):
    try:
        label_path = Path(f"data/labels/lablabel{item_no}.json")
        
        label_json = load_json(label_path)
        label_tests = flatten_label_json(label_json)

        pred_path = Path(f"{base_path}/paddlerows{item_no}.txt_rows.txt.json")

        pred_json = load_json(pred_path)
        pred_tests = flatten_label_json(pred_json['data'])

        if(pred_tests == []):
            return {"avg_score": 0, "duration": 0}


        evaluator = LabReportEvaluator(
            paths=["test_name", "result", "unit", "reference_value", "section"]
        )


        result = evaluator.evaluate(label_tests, pred_tests)
        return {"avg_score": result["avg_score"], "duration": pred_json['duration']}

    except:
        return {"avg_score": -1, "duration": -1}
    

In [7]:
res = calc_score(1, "outputs/results/gemma-4-e2b-it/row2json1")
res

{'avg_score': 84.82565217391306, 'duration': 41.8}

In [8]:
res = calc_score(1, "outputs/results/Qwen3-VL-4B-Instruct/row2json1")
res

{'avg_score': 77.78347826086956, 'duration': 36.39}

In [9]:
res

{'avg_score': 77.78347826086956, 'duration': 36.39}

In [10]:
models = ["q21w3e", "213"]
for model in models:
    print(model)

q21w3e
213


In [11]:
import os

path = 'outputs/results'
folders = [f for f in os.listdir(path) if os.path.isdir(os.path.join(path, f))]
print(folders)

['gemma-3-4b-it', 'gemma-4-e2b-it', 'gemma-4-e4b-it', 'granite-4.1-3b', 'granite-4.1-8b', 'Qwen3-VL-4B-Instruct']


In [12]:
import numpy as np
import os


path_models = 'outputs/results'
models = [f for f in os.listdir(path) if os.path.isdir(os.path.join(path, f))]

prompts = ['row2json1', 'row2json2', 'row2json3', 'row2json4', 'row2json5']
all_stat = {}

for model in models:
    prompt_stat = {}
    for prompt in prompts:
        scores = []
        durations = []
        
        for i in range(1, 101):
            res = calc_score(i, f"outputs/results/{model}/{prompt}")
            scores.append(res['avg_score'])
            durations.append(res['duration'])
        
        # Calculate stats AFTER collecting all 100 iterations
        stat = {
            "scores_mean": np.mean(scores), 
            "scores_nonezero_mean": np.mean([s for s in scores if s != 0]) if any(s != 0 for s in scores) else 0,
            "count_nonzero":  len([sa for sa in scores if sa == -1]),
            "durations_mean": np.mean([d for d in durations if d != 0]) if any(d != 0 for d in durations) else 0
        }
        prompt_stat[prompt] = stat
    all_stat[model] = prompt_stat

In [ ]:
len([sa for sa in scores if sa == -1])

20

: 

In [40]:
scores_mean_only = {}
for model, prompts_dict in all_stat.items():
    scores_mean_only[model] = {prompt: stats['count_nonzero'] for prompt, stats in prompts_dict.items()}

In [41]:
scores_mean_only

{'gemma-3-4b-it': {'row2json1': 16,
  'row2json2': 28,
  'row2json3': 35,
  'row2json4': 42,
  'row2json5': 41},
 'gemma-4-e2b-it': {'row2json1': 12,
  'row2json2': 100,
  'row2json3': 28,
  'row2json4': 26,
  'row2json5': 25},
 'gemma-4-e4b-it': {'row2json1': 18,
  'row2json2': 100,
  'row2json3': 32,
  'row2json4': 36,
  'row2json5': 39},
 'granite-4.1-3b': {'row2json1': 5,
  'row2json2': 5,
  'row2json3': 20,
  'row2json4': 34,
  'row2json5': 28},
 'granite-4.1-8b': {'row2json1': 6,
  'row2json2': 24,
  'row2json3': 17,
  'row2json4': 100,
  'row2json5': 24},
 'Qwen3-VL-4B-Instruct': {'row2json1': 3,
  'row2json2': 100,
  'row2json3': 12,
  'row2json4': 9,
  'row2json5': 20}}

In [42]:
import pandas as pd

df = pd.DataFrame.from_dict(scores_mean_only)
df

,gemma-3-4b-it,gemma-4-e2b-it,gemma-4-e4b-it,granite-4.1-3b,granite-4.1-8b,Qwen3-VL-4B-Instruct
row2json1,16,12,18,5,6,3
row2json2,28,100,100,5,24,100
row2json3,35,28,32,20,17,12
row2json4,42,26,36,34,100,9
row2json5,41,25,39,28,24,20


,gemma-3-4b-it,granite-4.1-3b,Qwen3-VL-4B-Instruct
row2json1,73.256018,73.162296,74.080443
row2json2,77.636051,73.817977,74.080443
row2json3,77.785986,74.422819,74.080443
row2json4,76.937722,73.951279,74.080443
row2json5,76.475405,74.080443,74.080443


In [ ]:

print(np.mean(scores), np.mean([s for s in scores if s != 0]))

53.46251340579711 75.29931465605227


In [112]:
import numpy as np

print(np.mean(durations), np.mean([s for s in durations if s != 0]))

190.40525252525248 269.28742857142856
